In [10]:
import pandas as pd
import numpy as np

# קליטת הקובץ
file_path = 'listings.csv'

# טעינת הדאטה
df = pd.read_csv(file_path)

###############

# הסרת שורות שבהן כל הערכים ריקים
df_clean = df.dropna(how='all').copy()

print(f"Shape after dropping empty rows: {df_clean.shape}")

###############

# פונקציה לניקוי המחיר והמרה למספר
def clean_price(value):
    if pd.isna(value):
        return np.nan
    # המרה למחרוזת, הסרת כל תו שאינו מספר (כמו ₪, פסיקים, רווחים)
    clean_val = str(value).replace(',', '').replace('₪', '').replace(' ', '').strip()
    # ניסיון המרה למספר
    try:
        return float(clean_val)
    except ValueError:
        return np.nan

# החלת הניקוי
df_clean['price'] = df_clean['price'].apply(clean_price)

###############

# סינון מחירים בטווח 400,000 עד 7,000,000 והסרת ערכים חסרים
df_clean = df_clean[
    (df_clean['price'] >= 400000) &
    (df_clean['price'] <= 7000000) &
    (df_clean['price'].notna())
]

# המרה למספרים שלמים (ללא נקודה עשרונית)
df_clean['price'] = df_clean['price'].astype(int)

print(f"Shape after price filter: {df_clean.shape}")
print(df_clean['price'].head())

##############

# המרה של שדה המ"ר למספרי (מטפל במקרים של טקסט בטעות)
df_clean['square_meters'] = pd.to_numeric(df_clean['square_meters'], errors='coerce')

# סינון: גודל בין 25 ל-1000
df_clean = df_clean[
    (df_clean['square_meters'] >= 25) &
    (df_clean['square_meters'] <= 1000)
]

print(f"Shape after sq meter filter: {df_clean.shape}")


###################################

# חישוב מחיר למטר והמרה למספר שלם
df_clean['price_per_meter'] = (df_clean['price'] / df_clean['square_meters']).astype(int)

print(df_clean[['price', 'square_meters', 'price_per_meter']].head())

###############################

def classify_media(row):
    # בדיקה האם יש תמונות (גדול מ-0 ולא NaN)
    has_images = pd.notna(row['images_count']) and row['images_count'] > 0

    # בדיקה האם יש וידאו (באחד משני השדות)
    has_video = (pd.notna(row['mp4_video_url']) and str(row['mp4_video_url']).strip() != '') or \
                (pd.notna(row['video_url']) and str(row['video_url']).strip() != '')

    if has_images and has_video:
        return 'גם תמונות וגם וידאו'
    elif has_images:
        return 'רק תמונות'
    elif has_video:
        return 'רק וידיאו'
    else:
        return 'ללא מדיה'

# הפעלת הפונקציה על כל שורה (axis=1)
df_clean['media_type'] = df_clean.apply(classify_media, axis=1)

##########################

# 1. יצירת מילון מיפוי בין קוד שכונה לשם שכונה מתוך הרשומות המלאות
hood_mapping = df_clean.dropna(subset=['neighborhood']).set_index('hood_id')['neighborhood'].to_dict()

# 2. פונקציה למילוי הערך החסר
def fill_neighborhood(row):
    # אם השם קיים, נשאיר אותו
    if pd.notna(row['neighborhood']):
        return row['neighborhood']
    # אחרת, ננסה למצוא לפי ה-id
    return hood_mapping.get(row['hood_id'], row['neighborhood']) # אם לא נמצא, נשאר NaN

# ביצוע ההשלמה
df_clean['neighborhood'] = df_clean.apply(fill_neighborhood, axis=1)

#####################

# רשימת ערכים להסרה
types_to_remove = [
    "מגרשים",
    "בית פרטי/ קוטג'",
    "דו משפחתי",
    "כללי",
    "חניה"
]

# סינון - משאירים רק את מה ש*לא* ברשימה הזו
df_clean = df_clean[~df_clean['HomeTypeID_text'].isin(types_to_remove)]

print("Home Types remaining:")
print(df_clean['HomeTypeID_text'].unique())
print(f"Final shape: {df_clean.shape}")

####################

Shape after dropping empty rows: (3647, 52)
Shape after price filter: (3427, 52)
0     599000
1    1890000
2    1030000
3     990000
4    1350000
Name: price, dtype: int64
Shape after sq meter filter: (3417, 52)
     price  square_meters  price_per_meter
0   599000          987.0              606
1  1890000          177.0            10677
2  1030000           90.0            11444
3   990000           76.0            13026
4  1350000           94.0            14361
Home Types remaining:
['דירה' 'גג/ פנטהאוז' 'דירת גן' 'דופלקס' 'יחידת דיור' 'טריפלקס']
Final shape: (2896, 54)


In [11]:
import plotly.express as px
import ast
import numpy as np

# --- חילוץ קורדינאטות  ---

if 'lat' not in df_clean.columns or 'lon' not in df_clean.columns:
    def parse_coordinates_dict(coord_str):
        try:
            if pd.isna(coord_str):
                return np.nan, np.nan
            coord_dict = ast.literal_eval(str(coord_str))
            lat = coord_dict.get('latitude')
            lon = coord_dict.get('longitude')
            return float(lat), float(lon)
        except:
            return np.nan, np.nan

    df_clean['lat'], df_clean['lon'] = zip(*df_clean['coordinates'].apply(parse_coordinates_dict))

# סינון שורות ללא מיקום תקין
df_map = df_clean.dropna(subset=['lat', 'lon']).copy()

# --- חישוב גבולות צבע חכמים (אחוזונים 5 ו-95) ---

p5_ppm = df_map['price_per_meter'].quantile(0.05)
p95_ppm = df_map['price_per_meter'].quantile(0.95)


# ---  Carto Voyager + סקאלת כחולים
fig_voyager = px.scatter_mapbox(
    df_map,
    lat="lat",
    lon="lon",
    color="price_per_meter",
    color_continuous_scale=[
        [0.0, "rgb(130, 200, 255)"],
        [1.0, "rgb(0, 0, 150)"]
    ],
    range_color=[p5_ppm, p95_ppm],
    opacity=0.7,
    hover_data={'neighborhood': True, 'square_meters': True, 'price_per_meter': ':.0f', 'lat': False, 'lon': False},
    zoom=12,
    height=600,
    width=600,
    title="מפת נכסים - סגנון Voyager (צבעי פסטל) + סקאלת כחולים"
)

fig_voyager.update_layout(
    mapbox_style="white-bg",
    mapbox_layers=[
        {
            "below": 'traces',
            "sourcetype": "raster",
            "sourceattribution": "CartoDB Voyager",
            "source": [
                "https://basemaps.cartocdn.com/rastertiles/voyager/{z}/{x}/{y}{r}.png"
            ]
        }
    ],
    margin={"r":0,"t":50,"l":0,"b":0}
)

fig_voyager.show()

In [12]:
import plotly.express as px
import pandas as pd

# 1. סינון ראשוני: מינימום 15 נכסים בשכונה
min_listings = 15
neighborhood_counts = df_map['neighborhood'].value_counts()
valid_neighborhoods_initial = neighborhood_counts[neighborhood_counts >= min_listings].index
df_temp = df_map[df_map['neighborhood'].isin(valid_neighborhoods_initial)].copy()

# 2. חישוב חציון ומיון (מהזול ליקר)
median_prices = df_temp.groupby('neighborhood')['price'].median().sort_values()

# 3. בחירת ה-Top 5 וה-Bottom 5
if len(median_prices) > 10:
    bottom_5 = median_prices.head(5).index.tolist()
    top_5 = median_prices.tail(5).index.tolist()
    selected_neighborhoods = bottom_5 + top_5
else:
    selected_neighborhoods = median_prices.index.tolist()

# זה מוריד את עסקאות הקיצון שמותחות את הגרף
filtered_dfs = []
for hood in selected_neighborhoods:
    subset = df_temp[df_temp['neighborhood'] == hood]
    p05 = subset['price'].quantile(0.05) # גבול תחתון
    p95 = subset['price'].quantile(0.95) # גבול עליון
    clean_subset = subset[(subset['price'] >= p05) & (subset['price'] <= p95)]
    filtered_dfs.append(clean_subset)

# איחוד הנתונים הנקיים חזרה לדאטה-פריים אחד
df_ridge_clean = pd.concat(filtered_dfs)

print(f"יוצר גרף Ridge נקי (ללא קיצון) עבור {len(selected_neighborhoods)} שכונות")

# --- יצירת גרף Ridgeline ---
fig_ridge = px.violin(
    df_ridge_clean,
    x='price',
    y='neighborhood',
    color='neighborhood',
    orientation='h',
    title='התפלגות מחירים לפי שכונות (ללא עסקאות קיצון)',
    labels={'neighborhood': 'שכונה', 'price': 'מחיר בש"ח'},
    category_orders={'neighborhood': selected_neighborhoods},
    color_discrete_sequence=px.colors.qualitative.Prism,
    box=False,
    points=False
)

# --- עיצוב ה-Ridge ---
fig_ridge.update_traces(
    side='positive',
    width=2.5,
    meanline_visible=True,
    line=dict(color='DarkSlateGrey', width=1.5),
    opacity=0.8

)

fig_ridge.update_layout(
    template="plotly_white",
    showlegend=False,
    xaxis_tickformat = ',.0f',
    height=700,
    violinmode='overlay',
    yaxis=dict(
        showgrid=True,
        title=None,
        tickfont=dict(size=12)
    ),
    xaxis=dict(
        title="מחיר נכס (₪)",
        zeroline=False
    ),
    margin=dict(l=150, t=60, b=50, r=50)
)

fig_ridge.show()

יוצר גרף Ridge נקי (ללא קיצון) עבור 10 שכונות


In [13]:
import plotly.express as px
import numpy as np
import pandas as pd

# 1. עיבוד וניקוי
df_rooms = df_map.copy()
df_rooms = df_rooms.dropna(subset=['line_1'])
df_rooms['rooms_extracted'] = df_rooms['line_1'].astype(str).str.extract(r'(\d+(?:\.\d+)?)').astype(float)
df_rooms['rooms_clean'] = np.floor(df_rooms['rooms_extracted'])
df_rooms = df_rooms[df_rooms['rooms_clean'].isin([2, 3, 4, 5])]

# 2. סינון שכונות
neighborhood_counts = df_rooms['neighborhood'].value_counts()
valid_hoods = neighborhood_counts[neighborhood_counts >= 20].index
df_filtered = df_rooms[df_rooms['neighborhood'].isin(valid_hoods)].copy()

# 3. חישוב אחוזים
df_counts = df_filtered.groupby(['neighborhood', 'rooms_clean']).size().reset_index(name='count')
total_per_hood = df_counts.groupby('neighborhood')['count'].transform('sum')
df_counts['percentage'] = (df_counts['count'] / total_per_hood) * 100

# --- לוגיקה להסתרת טקסט ---
# אם האחוז קטן מ-6%, הטקסט יהיה ריק. אחרת, הוא יהיה המספר עם %
df_counts['label_text'] = df_counts.apply(
    lambda row: f"{int(round(row['percentage']))}%" if row['percentage'] >= 6 else "",
    axis=1
)

df_counts['rooms_label'] = df_counts['rooms_clean'].astype(int).astype(str) + " חדרים"

# 4. מיון
small_apts_share = df_counts[df_counts['rooms_clean'].isin([2, 3])].groupby('neighborhood')['percentage'].sum()
sorted_neighborhoods = small_apts_share.sort_values(ascending=False).index.tolist()
remaining_hoods = [h for h in valid_hoods if h not in sorted_neighborhoods]
sorted_neighborhoods.extend(remaining_hoods)

# --- הגדרת צבעים פסטליים ---
pastel_colors = {
    "2 חדרים": "#BFD7ED", # תכלת פסטל בהיר מאוד
    "3 חדרים": "#96C0CE", # תכלת אפרפר
    "4 חדרים": "#60A3D9", # תכלת בינוני רך
    "5 חדרים": "#0074B7"  # כחול רך אך ברור
}

# 5. יצירת הגרף
fig_stack = px.bar(
    df_counts,
    x="neighborhood",
    y="percentage",
    color="rooms_label",
    text="label_text",
    title="התפלגות גודל דירות בשכונות (2-5 חדרים)",
    labels={"neighborhood": "שכונה", "percentage": "אחוז מסך הנכסים", "rooms_label": ""},

    category_orders={
        "neighborhood": sorted_neighborhoods,
        "rooms_label": ["2 חדרים", "3 חדרים", "4 חדרים", "5 חדרים"]
    },

    color_discrete_map=pastel_colors
)

# --- עיצוב ---
fig_stack.update_layout(
    template="plotly_white",
    height=600,
    bargap=0.2,

    yaxis=dict(
        title="אחוז מהנכסים (%)",
        ticksuffix="%",
        showgrid=True,
        gridcolor='#F0F0F0', # גריד אפור בהיר מאוד ועדין
        zeroline=False
    ),

    xaxis=dict(
        title=None,
        tickangle=-45
    ),

    # מיקום המקרא למעלה
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        title=None
    ),

    margin=dict(t=80, b=100),
    font=dict(family="Arial", size=12)
)

fig_stack.update_traces(
    textposition='inside',
    textfont=dict(size=13, color='white', family="Arial"),
    marker_line_color='white',
    marker_line_width=0.5,
    insidetextanchor='middle'
)

fig_stack.show()

In [14]:
import plotly.express as px
import pandas as pd
import numpy as np
import statsmodels.api as sm # נדרש עבור חישוב קו המגמה (ols)

# 1. הכנת הנתונים
df_analysis = df_map.copy()
if 'rooms_clean' not in df_analysis.columns:
    df_analysis['rooms_extracted'] = df_analysis['line_1'].astype(str).str.extract(r'(\d+(?:\.\d+)?)').astype(float)
    df_analysis['rooms_clean'] = np.floor(df_analysis['rooms_extracted'])

df_analysis = df_analysis[
    (df_analysis['price'] > 500000) &
    (df_analysis['price'] < 5000000) &
    (df_analysis['rooms_clean'].between(2, 6))
]

# סינון וחישוב חציונים
df_trend = df_analysis[df_analysis['images_count'] <= 16]
trend_data = df_trend.groupby('images_count')['price'].median().reset_index()

# 2. יצירת הגרף המשופר
fig_price = px.scatter(
    trend_data,
    x='images_count',
    y='price',
    trendline="ols", # קו מגמה לינארי
    trendline_color_override="black", # צבע בולט לקו המגמה
    title="הקשר בין השקעה בנראות (מספר תמונות) למחיר הנכס",
    labels={'images_count': 'מספר תמונות במודעה', 'price': 'מחיר חציוני (₪)'}
)

# 3. עיצוב הנראות
fig_price.update_traces(
    marker=dict(
        size=12,
        color='RoyalBlue',
    ),
    selector=dict(mode='markers') # מחיל את העיצוב רק על הנקודות ולא על הקו
)

# עיצוב מיוחד לקו המגמה
fig_price.data[1].line.dash = 'dash'
fig_price.data[1].line.width = 3

fig_price.update_layout(
    template="plotly_white",
    height=600,

    # עיצוב ציר ה-Y (מחיר)
    yaxis=dict(
        tickformat=',.0f', # פסיקים באלפים
        title_font=dict(size=14),
        showgrid=True,
        gridcolor='#F0F0F0'
    ),

    xaxis=dict(
        tickmode='linear', # מכריח את הציר להציג כל מספר
        dtick=1,           # קפיצות של 1 בדיוק
        showgrid=False,
        title_font=dict(size=14)
    ),

    # כותרת
    title=dict(
        font=dict(size=20),
        x=0.5, # מירכוז כותרת
        xanchor='center'
    ),

    # הוספת הסבר מרחף (Tooltip) נקי
    hovermode="x unified"
)

fig_price.show()

In [16]:
import plotly.express as px
import pandas as pd
import numpy as np

# 1. חישוב מחיר "מצופה" (Baseline)

base_prices = df_analysis.groupby(['neighborhood', 'rooms_clean'])['price'].mean().reset_index()
base_prices.rename(columns={'price': 'expected_price'}, inplace=True)

df_premium = df_analysis.merge(base_prices, on=['neighborhood', 'rooms_clean'], how='left')
df_premium['price_premium_percent'] = ((df_premium['price'] - df_premium['expected_price']) / df_premium['expected_price']) * 100

premium_trend = df_premium.groupby('images_count')['price_premium_percent'].mean().reset_index()
premium_trend_filtered = premium_trend[premium_trend['images_count'] <= 15].copy()

pastel_diverging = ["#EF5350", "#F5F5F5", "#66BB6A"]

# 2. הכנת טקסט להצגה (עם סימן %)
premium_trend_filtered['label_text'] = premium_trend_filtered['price_premium_percent'].apply(lambda x: f"{x:+.1f}%")

# 3. יצירת הגרף המשופר
fig_premium = px.bar(
    premium_trend_filtered,
    x='images_count',
    y='price_premium_percent',
    title='כמה שווה תמונה? הפער מהמחיר הממוצע בשכונה',
    labels={'images_count': 'מספר תמונות במודעה', 'price_premium_percent': 'סטייה מהמחיר הממוצע (%)'},

    # צביעה לפי הערך
    color='price_premium_percent',

    # סקאלת צבעים: אדום (שלילי) -> צהוב (אפס) -> ירוק (חיובי)
    color_continuous_scale=pastel_diverging,
    color_continuous_midpoint=0,

    text='label_text'
)

# 4. עיצוב ושיפורים ויזואליים
fig_premium.update_traces(
    textposition='outside', # הטקסט מעל/מתחת לבר ולא בתוכו
    marker_line_color='black', # מסגרת דקה לכל בר
    marker_line_width=1,
    textfont_size=12
)

fig_premium.update_layout(
    template="plotly_white",
    height=550,

    # הסתרת בר הצבעים בצד (כי הצבעים אינטואיטיביים)
    coloraxis_showscale=False,

    yaxis=dict(
        title="סטייה באחוזים מהמחיר הממוצע",
        ticksuffix="%", # מוסיף % לציר עצמו
        showgrid=True,
        gridcolor='#E0E0E0',
        zeroline=True,      # קו האפס המובנה של פלוטלי
        zerolinecolor='black', # צבע קו האפס
        zerolinewidth=2     # עובי קו האפס
    ),

    xaxis=dict(
        title="מספר תמונות",
        tickmode='linear', # מספרים שלמים בלבד
        dtick=1,
        showgrid=False
    ),

    # הרחבת הטווח בציר ה-Y כדי שהטקסט (outside) לא ייחתך
    margin=dict(t=80, b=50)
)

fig_premium.add_annotation(
    x=2, y=premium_trend_filtered['price_premium_percent'].min(),
    text="מתחת לממוצע", showarrow=False, yshift=-20, font=dict(color="red")
)
fig_premium.add_annotation(
    x=13, y=premium_trend_filtered['price_premium_percent'].max(),
    text="מעל הממוצע", showarrow=False, yshift=20, font=dict(color="green")
)

fig_premium.show()